## Text input

In [ ]:
from langchain.agents import create_agent
from azure_openai_llm import create_azure_llm
from langchain.messages import HumanMessage

from dotenv import load_dotenv

load_dotenv()

llm = create_azure_llm()

agent = create_agent(llm,
    system_prompt="You are a science fiction writer, create a capital city at the users request.",
)

In [ ]:
question = HumanMessage(content=[
    {"type": "text", "text": "What is the capital of The Moon?"}
])

response = agent.invoke(
    {"messages": [question]}
)

print(response['messages'][-1].content)

## Image input

In [ ]:
from ipywidgets import FileUpload
from IPython.display import display

uploader = FileUpload(accept='.png', multiple=False)
display(uploader)
print(uploader.value)

In [ ]:
import base64

# Get the first (and only) uploaded file dict
uploaded_file = uploader.value[0]

# This is a memoryview
content_mv = uploaded_file["content"]

# Convert memoryview -> bytes
img_bytes = bytes(content_mv)  # or content_mv.tobytes()

# Now base64 encode
img_b64 = base64.b64encode(img_bytes).decode("utf-8")

In [ ]:
multimodal_question = HumanMessage(content=[
    {"type": "text", "text": "Tell me about this diagram"},
    {"type": "image", "base64": img_b64, "mime_type": "image/png"}
])

response = agent.invoke(
    {"messages": [multimodal_question]}
)

print(response['messages'][-1].content)

In [3]:
import sounddevice as sd
from scipy.io.wavfile import write
import base64
import io
import time
from tqdm import tqdm

# Recording settings
duration = 3  # seconds
sample_rate = 44100

print("Recording...")
audio = sd.rec(int(duration * sample_rate), samplerate=sample_rate, channels=1)
# Progress bar for the duration
for _ in tqdm(range(duration * 10)):   # update 10× per second
    time.sleep(0.1)
sd.wait()
print("Done.")

# Write WAV to an in-memory buffer
buf = io.BytesIO()
write(buf, sample_rate, audio)
wav_bytes = buf.getvalue()
aud_b64 = base64.b64encode(wav_bytes).decode("utf-8")

Recording...


100%|██████████| 30/30 [00:03<00:00,  9.87it/s]


Done.


In [4]:
# Check what deployments you have available
import os
from dotenv import load_dotenv

load_dotenv()

print("🔍 Current Azure AI Foundry Configuration:")
print("=" * 50)
print(f"Endpoint: {os.getenv('AZURE_OPENAI_ENDPOINT')}")
print(f"Chat Deployment: {os.getenv('AZURE_OPENAI_CHAT_DEPLOYMENT')}")
print(f"Audio Deployment: {os.getenv('AZURE_OPENAI_AUDIO_DEPLOYMENT')}")
print(f"API Version: {os.getenv('AZURE_OPENAI_API_VERSION')}")
print("=" * 50)

# Test if we can create connections to both deployments
print("\n📋 Testing Model Availability:")
print("-" * 30)

try:
    from azure_openai_llm import create_azure_llm
    
    print("✅ Chat model connection:")
    chat_llm = create_azure_llm(type="chat")
    print(f"   Model type: {type(chat_llm).__name__}")
    
    print("\n✅ Audio model connection:")  
    audio_llm = create_azure_llm(type="audio")
    print(f"   Model type: {type(audio_llm).__name__}")
    
except Exception as e:
    print(f"❌ Error: {e}")

🔍 Current Azure AI Foundry Configuration:
Endpoint: https://aoai-foundry-swc.openai.azure.com/
Chat Deployment: lums-gpt-4.1-mini
Audio Deployment: lums-gpt-audio-mini
API Version: 2025-01-01-preview

📋 Testing Model Availability:
------------------------------
✅ Chat model connection:
   Model type: AzureChatOpenAI

✅ Audio model connection:
   Model type: AzureChatOpenAI


In [ ]:
from langchain.agents import create_agent
from azure_openai_llm import create_azure_llm
from langchain.messages import HumanMessage

agent = create_agent(create_azure_llm(type="audio", api="openai"))

multimodal_question = HumanMessage(content=[
    {"type": "text", "text": "Tell me about this audio file"},
    {"type": "audio", "base64": aud_b64, "mime_type": "audio/wav"}
])

response = agent.invoke(
    {"messages": [multimodal_question]}
)

print(response['messages'][-1].content)

=== Testing Audio-Capable Models in Azure AI Foundry ===

1️⃣ Using dedicated audio model (lums-gpt-audio-mini):
✅ Audio model response:
I'm sorry, but I can't analyze or listen to audio files directly. If you can provide a text description or transcription of what you hear in the audio, I’ll do my best to help you analyze it. Let me know how you'd like to proceed.


2️⃣ Fallback using regular chat model (lums-gpt-4.1-mini):
❌ Chat model with audio failed: Error code: 400 - {'error': {'message': "Invalid 'messages[0]'. Content blocks are expected to be either text or image_url type.", 'type': 'invalid_request_error', 'param': 'messages[0]', 'code': 'invalid_value'}}
This model may not support multimodal audio input
